# Migrated from Dataiku

**Original recipe:** `build_customer_features`

**Source project:** `CUSTOMER_ANALYTICS`

**Migration date:** `2026-03-24`

**Review flags:** None

In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

In [ ]:
# Input: Load datasets
df_CUSTOMER_360_VIEW = spark.read.format("delta").load("Tables/CUSTOMER_360_VIEW")
df_product_metrics = spark.read.format("delta").load("Tables/product_metrics")

In [ ]:
# Migrated logic
# import dataiku  # Removed — using PySpark native APIs
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Read input datasets
customer_df = spark.read.format("delta").load("Tables/CUSTOMER_360_VIEW")
product_df = spark.read.format("delta").load("Tables/product_metrics")

# Get project variables
project_vars = spark.conf.get("spark.custom_variables", "{}")  # Was dataiku.get_custom_variables()
lookback_days = int(project_vars.get('LOOKBACK_DAYS', 90))
min_orders = int(project_vars.get('MIN_ORDERS', 1))

# Feature engineering
customer_df['account_age_days'] = (datetime.now() - pd.to_datetime(customer_df['CREATED_DATE_STR'])).dt.days
customer_df['avg_order_value'] = customer_df['LIFETIME_REVENUE'] / customer_df['TOTAL_ORDERS'].replace(0, np.nan)
customer_df['orders_per_month'] = customer_df['TOTAL_ORDERS'] / (customer_df['account_age_days'] / 30.0)

# RFM scoring
customer_df['recency_score'] = pd.qcut(customer_df['account_age_days'], q=5, labels=[5,4,3,2,1])
customer_df['frequency_score'] = pd.qcut(customer_df['TOTAL_ORDERS'].rank(method='first'), q=5, labels=[1,2,3,4,5])
customer_df['monetary_score'] = pd.qcut(customer_df['LIFETIME_REVENUE'].rank(method='first'), q=5, labels=[1,2,3,4,5])

customer_df['rfm_score'] = (
    customer_df['recency_score'].astype(int) +
    customer_df['frequency_score'].astype(int) +
    customer_df['monetary_score'].astype(int)
)

# Segment assignment
def assign_segment(row):
    if row['rfm_score'] >= 12:
        return 'Champion'
    elif row['rfm_score'] >= 9:
        return 'Loyal'
    elif row['rfm_score'] >= 6:
        return 'Potential'
    elif row['rfm_score'] >= 3:
        return 'At Risk'
    else:
        return 'Hibernating'

customer_df['segment'] = customer_df.apply(assign_segment, axis=1)

# Save enriched features to managed folder for model training
folder = dataiku.Folder("model_training_data")
folder_path = folder.get_path()
customer_df.to_parquet(f"{folder_path}/customer_features.parquet", index=False)

# Write output dataset
output = dataiku.Dataset("customer_features")
output.write_dataframe(customer_df)

print(f"Processed {len(customer_df)} customers, {customer_df['segment'].value_counts().to_dict()}")


In [ ]:
# Output: Write results
df_customer_features.write.format("delta").mode("overwrite").save("Tables/customer_features")